# 2.0 - LBP + SVM

Three LBP feature configurations (see src/features.py for details and
citations), each trained with SVM, evaluated on validation. The best
configuration is then evaluated once on the test set for the final
reported metric.

In [ ]:
# Clone the repo and install dependencies.
# facenet-pytorch needs --no-deps: Colab has no prebuilt wheels for the
# old numpy/Pillow versions it normally asks for.

!git clone https://github.com/laianemuckler/liveness-detection.git
%cd liveness-detection

!pip install -r requirements.txt --quiet
!pip install facenet-pytorch==2.6.0 --no-deps --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Imports: paths from config, dataset loading, feature extraction,
# model training, and evaluation/logging.

import os
import numpy as np
from tqdm import tqdm

from src.config import TRAIN_DIR, VAL_DIR, TEST_DIR, DRIVE_ROOT
from src.dataset import load_processed_split
from src.features import extract_features
from src.modeling.train import train_svm
from src.modeling.predict import evaluate, log_experiment

In [ ]:
# Load the already-processed (MTCNN-aligned) image paths + labels
# for each split. This does NOT re-run MTCNN; it just lists the
# files saved by 01_preprocessing.

train_paths, y_train = load_processed_split(TRAIN_DIR)
val_paths, y_val = load_processed_split(VAL_DIR)
test_paths, y_test = load_processed_split(TEST_DIR)

print(f"Train: {len(train_paths)} | Validation: {len(val_paths)} | Test: {len(test_paths)}")

## Folder to save extracted features and results

Features are saved to Drive so they don't need to be recomputed if
this notebook is re-run (feature extraction is the slow part).

In [ ]:
RUNS_DIR = os.path.join(DRIVE_ROOT, 'experiments', 'lbp_svm', 'runs')
os.makedirs(RUNS_DIR, exist_ok=True)

## Helper: run one full experiment (extract -> train -> evaluate -> log)

This function is reused for all 3 LBP configurations below, so each
experiment follows the exact same steps and nothing is done by hand.

In [ ]:
def run_lbp_experiment(exp_id, feature_config_name, svm_kernel='rbf', svm_C=1.0):
    exp_dir = os.path.join(RUNS_DIR, exp_id)
    os.makedirs(exp_dir, exist_ok=True)

    # 1. Extract features (or load them if already saved from a previous run)
    train_feat_path = os.path.join(exp_dir, 'X_train.npy')
    val_feat_path = os.path.join(exp_dir, 'X_val.npy')
    test_feat_path = os.path.join(exp_dir, 'X_test.npy')

    if os.path.exists(train_feat_path):
        print(f"[{exp_id}] Features already extracted, loading from disk...")
        X_train = np.load(train_feat_path)
        X_val = np.load(val_feat_path)
        X_test = np.load(test_feat_path)
    else:
        print(f"[{exp_id}] Extracting '{feature_config_name}' features...")
        X_train = extract_features(train_paths, feature_config_name)
        X_val = extract_features(val_paths, feature_config_name)
        X_test = extract_features(test_paths, feature_config_name)

        np.save(train_feat_path, X_train)
        np.save(val_feat_path, X_val)
        np.save(test_feat_path, X_test)

    print(f"[{exp_id}] Feature shape: {X_train.shape}")

    # 2. Train SVM on the training set
    print(f"[{exp_id}] Training SVM...")
    model, scaler = train_svm(X_train, y_train, kernel=svm_kernel, C=svm_C)

    # 3. Evaluate on validation (used to compare configurations)
    val_results = evaluate(model, scaler, X_val, y_val)
    print(f"[{exp_id}] Validation -> HTER: {val_results['HTER']*100:.2f}% | AUC: {val_results['AUC']:.4f}")

    # 4. Log to experiment_log.csv (test metrics left empty for now)
    log_experiment(
        exp_id=exp_id,
        metodo='LBP+SVM',
        feature_config=feature_config_name,
        modelo_config=f"kernel={svm_kernel}, C={svm_C}",
        hter_val=val_results['HTER'],
        auc_val=val_results['AUC'],
    )

    return {
        'model': model,
        'scaler': scaler,
        'X_test': X_test,
        'val_results': val_results,
    }

## Experiment 1: LBP global (P=8, R=1)

Single histogram over the whole face. This is the configuration that
originally produced the broken result (identical scores for every
image) -- kept here for the historical record of the fix.

In [ ]:
exp1 = run_lbp_experiment('lbp_svm_01_global', 'lbp_global_p8r1')

## Experiment 2: LBP grid (P=8, R=1, 3x3 blocks)

Per-block features, as recommended by Chingovska et al. (2012)
specifically for the NUAA dataset.

In [ ]:
exp2 = run_lbp_experiment('lbp_svm_02_grid', 'lbp_grid_p8r1_3x3')

## Experiment 3: LBP Maatta-style (P=16,R=2 + P=8,R=1 blocks + P=8,R=1 global)

Approximate replication of the method compared against in
Chingovska et al. (originally Maatta, Hadid and Pietikainen, 2011).
See src/features.py docstring for the documented assumption about
block overlap (not specified in the original paper)

In [ ]:
exp3 = run_lbp_experiment('lbp_svm_03_maatta', 'lbp_maatta_p16r2_p8r1')

## Compare the 3 experiments on validation

In [ ]:
experiments = {
    'lbp_svm_01_global': exp1,
    'lbp_svm_02_grid': exp2,
    'lbp_svm_03_maatta': exp3,
}

for exp_id, exp in experiments.items():
    r = exp['val_results']
    print(f"{exp_id}: HTER={r['HTER']*100:.2f}% | AUC={r['AUC']:.4f}")

## Pick the best config and evaluate ONCE on the test set

IMPORTANT: only the winning configuration is evaluated on the test
set, and only once. This keeps the test metric an honest, unbiased
estimate (see the conversation history / docs/decisions.md for why).

Edit BEST_EXP_ID below after looking at the comparison above.

BEST_EXP_ID = 'lbp_svm_02_grid'  # <-- change this based on the validation comparison

best = experiments[BEST_EXP_ID]
test_results = evaluate(best['model'], best['scaler'], best['X_test'], y_test)

print(f"FINAL TEST RESULT ({BEST_EXP_ID})")
print(f"HTER: {test_results['HTER']*100:.2f}%")
print(f"AUC:  {test_results['AUC']:.4f}")

In [ ]:
# Update the log with the final test metrics for the winning config.
# Re-logs the same exp_id -- this simply appends a new row noting the
# test evaluation; the CSV keeps the full history.

log_experiment(
    exp_id=BEST_EXP_ID + '_FINAL_TEST',
    metodo='LBP+SVM',
    feature_config=BEST_EXP_ID,
    modelo_config='best config chosen on validation',
    hter_test=test_results['HTER'],
    auc_test=test_results['AUC'],
    obs='Final reported test metric for LBP+SVM'
)

## Save predictions for later analysis (error inspection, plots, etc.)

In [ ]:
import pandas as pd

df_predictions = pd.DataFrame({
    'image_path': test_paths,
    'label_true': y_test,
    'label_pred': test_results['y_pred'],
    'score': test_results['y_scores'],
})

predictions_path = os.path.join(RUNS_DIR, BEST_EXP_ID, 'predictions_test.csv')
df_predictions.to_csv(predictions_path, index=False)
print(f"Predictions saved to {predictions_path}")